# TUGAS MANDIRI

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, round as sround, count, avg

spark = SparkSession.builder \
    .appName("Tugas12-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

df_t = spark.read.csv("tugas12_ulasan.csv", header=True, inferSchema=True)
df_t.printSchema()
print("Jumlah baris:", df_t.count())
df_t.show(5)

26/11/19 14:03:44 WARN Utils: Your hostname, xcel resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/11/19 14:03:44 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/19 14:03:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


root
 |-- nama_menu: string (nullable = true)
 |-- cabang: string (nullable = true)
 |-- waktu_pesan: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- waktu_tunggu_menit: integer (nullable = true)
 |-- ulasan_positif: integer (nullable = true)

Jumlah baris: 3000
+--------------+----------+-----------+---+------------------+--------------+
|     nama_menu|    cabang|waktu_pesan|qty|waktu_tunggu_menit|ulasan_positif|
+--------------+----------+-----------+---+------------------+--------------+
|     Kopi Susu|Yogyakarta|       Sore|  5|                 5|             1|
|        Es Teh|Yogyakarta|       Pagi|  1|                21|             0|
|    Cappuccino|Yogyakarta|       Sore|  5|                15|             0|
|Kentang Goreng|  Semarang|      Malam|  2|                 5|             1|
|     Americano|Yogyakarta|      Malam|  2|                10|             0|
+--------------+----------+-----------+---+------------------+--------------+
only showing top 

### A. Membangun ML Pipeline

Urutan tahap: `StringIndexer` (3 kolom) → `OneHotEncoder` → `VectorAssembler` (hasil one-hot + `qty` + `waktu_tunggu_menit`) → `LogisticRegression`.

In [2]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml import Pipeline

# Tahap 1-3: StringIndexer untuk tiap kolom kategorikal
idx_menu = StringIndexer(inputCol="nama_menu", outputCol="menu_idx")
idx_cabang = StringIndexer(inputCol="cabang", outputCol="cabang_idx")
idx_waktu = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_idx")

# Tahap 4: OneHotEncoder untuk ketiga kolom indeks
ohe_t = OneHotEncoder(
    inputCols=["menu_idx", "cabang_idx", "waktu_idx"],
    outputCols=["menu_vec", "cabang_vec", "waktu_vec"]
)

# Tahap 5: VectorAssembler -- hasil one-hot + qty + waktu_tunggu_menit
asm_t = VectorAssembler(
    inputCols=["menu_vec", "cabang_vec", "waktu_vec", "qty", "waktu_tunggu_menit"],
    outputCol="features"
)

# Tahap 6: Logistic Regression
lr_t = LogisticRegression(featuresCol="features", labelCol="ulasan_positif")

pipeline_lr = Pipeline(stages=[idx_menu, idx_cabang, idx_waktu, ohe_t, asm_t, lr_t])
print("Pipeline Logistic Regression terdefinisi dengan", len(pipeline_lr.getStages()), "tahap.")

Pipeline Logistic Regression terdefinisi dengan 6 tahap.


### B. Pelatihan & Evaluasi

In [3]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

# Split 80:20 dengan seed=77
train_t, test_t = df_t.randomSplit([0.8, 0.2], seed=77)
print(f"Data training: {train_t.count()} | Data testing: {test_t.count()}")

# Latih pipeline pada data training
model_lr_t = pipeline_lr.fit(train_t)
pred_lr_t = model_lr_t.transform(test_t)

# Evaluator
ev_acc_t = MulticlassClassificationEvaluator(labelCol="ulasan_positif", predictionCol="prediction", metricName="accuracy")
ev_f1_t = MulticlassClassificationEvaluator(labelCol="ulasan_positif", predictionCol="prediction", metricName="f1")
ev_auc_t = BinaryClassificationEvaluator(labelCol="ulasan_positif", metricName="areaUnderROC")

acc_lr = ev_acc_t.evaluate(pred_lr_t)
f1_lr = ev_f1_t.evaluate(pred_lr_t)
auc_lr = ev_auc_t.evaluate(pred_lr_t)

print("=== Logistic Regression ===")
print(f"Accuracy : {acc_lr:.4f}")
print(f"F1-score : {f1_lr:.4f}")
print(f"AUC-ROC  : {auc_lr:.4f}")

print("\nConfusion matrix:")
pred_lr_t.groupBy("ulasan_positif", "prediction").count().orderBy("ulasan_positif", "prediction").show()

Data training: 2411 | Data testing: 589


=== Logistic Regression ===
Accuracy : 0.7929
F1-score : 0.7926
AUC-ROC  : 0.8650

Confusion matrix:
+--------------+----------+-----+
|ulasan_positif|prediction|count|
+--------------+----------+-----+
|             0|       0.0|  257|
|             0|       1.0|   57|
|             1|       0.0|   65|
|             1|       1.0|  210|
+--------------+----------+-----+



### C. Model Pembanding: Random Forest

Lima tahap feature engineering sama persis dengan bagian A, hanya tahap terakhir diganti dengan `RandomForestClassifier(numTrees=30)`.

In [4]:
rf_t = RandomForestClassifier(featuresCol="features", labelCol="ulasan_positif", numTrees=30)

pipeline_rf_t = Pipeline(stages=[idx_menu, idx_cabang, idx_waktu, ohe_t, asm_t, rf_t])
model_rf_t = pipeline_rf_t.fit(train_t)
pred_rf_t = model_rf_t.transform(test_t)

acc_rf = ev_acc_t.evaluate(pred_rf_t)
f1_rf = ev_f1_t.evaluate(pred_rf_t)
auc_rf = ev_auc_t.evaluate(pred_rf_t)

print(f"{'Model':<22}{'Accuracy':>10}{'F1':>10}{'AUC-ROC':>10}")
print(f"{'Logistic Regression':<22}{acc_lr:>10.4f}{f1_lr:>10.4f}{auc_lr:>10.4f}")
print(f"{'Random Forest':<22}{acc_rf:>10.4f}{f1_rf:>10.4f}{auc_rf:>10.4f}")

print("\nConfusion matrix Random Forest:")
pred_rf_t.groupBy("ulasan_positif", "prediction").count().orderBy("ulasan_positif", "prediction").show()

Model                   Accuracy        F1   AUC-ROC
Logistic Regression       0.7929    0.7926    0.8650
Random Forest             0.7963    0.7957    0.8456

Confusion matrix Random Forest:
+--------------+----------+-----+
|ulasan_positif|prediction|count|
+--------------+----------+-----+
|             0|       0.0|  262|
|             0|       1.0|   52|
|             1|       0.0|   68|
|             1|       1.0|  207|
+--------------+----------+-----+



**Perbandingan (bagian C).** Kedua model hasilnya sangat berdekatan: accuracy dan F1-score Random Forest hanya unggul sangat tipis (selisih kurang dari 0,01) dari Logistic Regression, sedangkan AUC-ROC justru lebih tinggi pada Logistic Regression (sekitar 0,865 berbanding 0,84). Artinya Random Forest tidak lebih baik dalam membedakan ulasan positif dan negatif, padahal modelnya lebih kompleks. Ini wajar karena pola datanya sederhana: peluang ulasan positif turun mengikuti `waktu_tunggu_menit` secara halus (kurva sigmoid), dan pola seperti itu memang sangat cocok dengan Logistic Regression. Random Forest yang memecah data menjadi banyak tebakan bertingkat justru kurang halus dalam menghasilkan probabilitas, sehingga AUC-nya sedikit di bawah. Karena itu Logistic Regression dipilih sebagai model terbaik untuk bagian D (berdasarkan AUC tertinggi).

### D. Analisis Kesalahan Model

Model terbaik dipilih otomatis berdasarkan AUC-ROC tertinggi antara bagian B dan C.

In [5]:
# Pilih model terbaik berdasarkan AUC
if auc_lr >= auc_rf:
    nama_terbaik, pred_terbaik, model_terbaik = "Logistic Regression", pred_lr_t, model_lr_t
else:
    nama_terbaik, pred_terbaik, model_terbaik = "Random Forest", pred_rf_t, model_rf_t
print("Model terbaik (AUC tertinggi):", nama_terbaik)

# 10 baris di mana model SALAH memprediksi
salah = pred_terbaik.filter(col("ulasan_positif") != col("prediction"))
print("Total prediksi salah:", salah.count(), "dari", pred_terbaik.count())

salah.select("nama_menu", "cabang", "waktu_pesan", "qty", "waktu_tunggu_menit",
             "ulasan_positif", "prediction", "probability").show(10, truncate=False)

Model terbaik (AUC tertinggi): Logistic Regression
Total prediksi salah: 122 dari 589
+---------+--------+-----------+---+------------------+--------------+----------+----------------------------------------+
|nama_menu|cabang  |waktu_pesan|qty|waktu_tunggu_menit|ulasan_positif|prediction|probability                             |
+---------+--------+-----------+---+------------------+--------------+----------+----------------------------------------+
|Americano|Magelang|Malam      |1  |15                |1             |0.0       |[0.6253879175414343,0.3746120824585657] |
|Americano|Magelang|Siang      |2  |22                |1             |0.0       |[0.9113822566149014,0.08861774338509865]|
|Americano|Magelang|Sore       |1  |7                 |0             |1.0       |[0.23209137939008387,0.7679086206099162]|
|Americano|Magelang|Sore       |2  |11                |0             |1.0       |[0.42615454016905935,0.5738454598309406]|
|Americano|Magelang|Sore       |3  |5                

In [6]:
# Pendukung analisis pola: tingkat kesalahan per rentang waktu tunggu
pred_terbaik.withColumn(
    "rentang_tunggu",
    when(col("waktu_tunggu_menit") <= 6, "01. 2-6 menit")
    .when(col("waktu_tunggu_menit") <= 10, "02. 7-10 menit")
    .when(col("waktu_tunggu_menit") <= 14, "03. 11-14 menit")
    .when(col("waktu_tunggu_menit") <= 18, "04. 15-18 menit")
    .otherwise("05. 19-24 menit")
).groupBy("rentang_tunggu").agg(
    count("*").alias("jumlah"),
    sround(avg((col("ulasan_positif") != col("prediction")).cast("int")) * 100, 1).alias("persen_salah")
).orderBy("rentang_tunggu").show()

+---------------+------+------------+
| rentang_tunggu|jumlah|persen_salah|
+---------------+------+------------+
|  01. 2-6 menit|   134|        10.4|
| 02. 7-10 menit|   105|        24.8|
|03. 11-14 menit|    92|        47.8|
|04. 15-18 menit|    88|        25.0|
|05. 19-24 menit|   170|         9.4|
+---------------+------+------------+



In [7]:
# Pendukung E: fitur paling berpengaruh
meta = pred_terbaik.schema["features"].metadata["ml_attr"]["attrs"]
fitur = [None] * sum(len(v) for v in meta.values())
for tipe in meta:
    for f in meta[tipe]:
        fitur[f["idx"]] = f["name"]

print("Koefisien Logistic Regression (diurutkan dari paling berpengaruh):")
koef = list(zip(fitur, model_lr_t.stages[-1].coefficients.toArray()))
for nama, k in sorted(koef, key=lambda x: -abs(x[1])):
    print(f"  {nama:<28}{k:>9.4f}")

print("\nFeature importance Random Forest:")
imp = list(zip(fitur, model_rf_t.stages[-1].featureImportances.toArray()))
for nama, v in sorted(imp, key=lambda x: -x[1])[:5]:
    print(f"  {nama:<28}{v:>9.4f}")

Koefisien Logistic Regression (diurutkan dari paling berpengaruh):
  waktu_vec_Malam                0.2530
  waktu_tunggu_menit            -0.2220
  waktu_vec_Sore                 0.1857
  menu_vec_Roti Bakar           -0.1443
  menu_vec_Kopi Susu             0.1171
  cabang_vec_Semarang           -0.1140
  menu_vec_Americano            -0.0855
  waktu_vec_Pagi                -0.0651
  cabang_vec_Yogyakarta          0.0645
  menu_vec_Cappuccino            0.0194
  menu_vec_Es Teh               -0.0191
  qty                           -0.0108

Feature importance Random Forest:
  waktu_tunggu_menit             0.9397
  qty                            0.0125
  cabang_vec_Yogyakarta          0.0075
  waktu_vec_Pagi                 0.0061
  waktu_vec_Malam                0.0056


### E. Kesimpulan Bisnis

**Rekomendasi untuk manajemen Kopi Nusantara**

Model Logistic Regression yang dibangun mencapai accuracy sekitar 79% dan AUC-ROC sekitar 0,87, dan temuan paling jelas dari model adalah bahwa **waktu tunggu adalah faktor yang paling menentukan** apakah pelanggan memberi ulasan positif. Pada Random Forest, `waktu_tunggu_menit` menyumbang sekitar 93% dari total feature importance, sedangkan koefisiennya pada Logistic Regression bernilai negatif, yaitu semakin lama pelanggan menunggu, semakin kecil peluang ulasan positif. Atribut lain (menu, cabang, waktu pesan, dan qty) pengaruhnya kecil dan tidak konsisten, jadi tidak layak dijadikan dasar keputusan besar.

Dari analisis kesalahan di bagian D, model paling sering salah pada pesanan dengan waktu tunggu sekitar 11-14 menit (tingkat kesalahan mendekati 50%), sedangkan pada waktu tunggu yang sangat singkat (di bawah 7 menit) atau sangat lama (di atas 18 menit) tingkat kesalahannya hanya sekitar 10%. Rentang 11-14 menit ini adalah zona abu-abu: pelanggan bisa puas atau kecewa dan model tidak dapat memastikannya dari atribut pesanan.

Rekomendasi konkret:
1. Tetapkan target waktu penyajian maksimal sekitar 10 menit untuk semua menu dan cabang, karena di atas 15 menit peluang ulasan positif turun tajam.
2. Pantau pesanan yang sudah menunggu 10-14 menit sebagai zona kritis, lalu beri intervensi cepat seperti prioritas penyajian atau permintaan maaf ringan, karena pada zona inilah kepuasan pelanggan masih bisa diselamatkan.
3. Fokuskan perbaikan operasional (jumlah barista, alur dapur) pada kemacetan waktu tunggu, bukan pada pergantian menu atau cabang, karena model tidak menunjukkan menu atau cabang tertentu yang bermasalah.
4. Untuk meningkatkan akurasi model berikutnya, tambahkan fitur yang belum ada, misalnya jumlah antrean saat memesan atau jumlah staf yang bertugas.

Catatan: data ini hasil simulasi, sehingga angka tepatnya dapat berbeda sedikit pada setiap kali dijalankan, tetapi pola utamanya (waktu tunggu sebagai faktor dominan) tetap sama.

In [8]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
